# 🧠 Module 4: RAG — Retrieval-Augmented Generation

**Goal:** Give your agent long-term memory using a vector database.

Steps:
1. Build a knowledge base from raw text
2. Chunk documents into small pieces
3. Embed chunks using a free local model
4. Store in ChromaDB (free vector database)
5. Query by similarity
6. Build the full RAG pipeline
7. (Optional) Use Groq LLM to generate grounded answers

> **No GPU needed. Runs on free Colab CPU.**

---
## Step 1: Install Dependencies

In [ ]:
!pip install -q sentence-transformers chromadb
print('✅ Dependencies installed')

---
## Step 2: Create a Knowledge Base

Imagine these are pages from a company policy document. In a real project, you would load PDFs or database records.

In [ ]:
# Our fake company documents
DOCUMENTS = [
    {
        "id": "doc_001",
        "source": "refund_policy.txt",
        "text": "Our return policy allows customers to return any product within 30 days of purchase. "
                "Items must be in their original condition with all original packaging. "
                "Digital products and software licenses cannot be returned once activated."
    },
    {
        "id": "doc_002",
        "source": "refund_policy.txt",
        "text": "To process a refund, customers must contact our support team at support@company.com. "
                "Refunds are processed within 5 to 7 business days. "
                "The refund will be credited to the original payment method."
    },
    {
        "id": "doc_003",
        "source": "shipping_policy.txt",
        "text": "Standard shipping takes 3 to 5 business days within the country. "
                "Express shipping is available for an additional fee and delivers within 1 to 2 days. "
                "International shipping takes 7 to 14 business days."
    },
    {
        "id": "doc_004",
        "source": "shipping_policy.txt",
        "text": "Free shipping is offered on all orders above $50. "
                "Orders are dispatched within 24 hours of payment confirmation. "
                "Tracking information is sent to the customer's email once the order is shipped."
    },
    {
        "id": "doc_005",
        "source": "pricing_policy.txt",
        "text": "Our pricing is inclusive of all taxes. "
                "We offer a price match guarantee: if you find the same product cheaper elsewhere, "
                "we will match that price. Contact sales@company.com with proof of the lower price."
    },
]

print(f'✅ Loaded {len(DOCUMENTS)} documents')
print('Sources:', list(set(d['source'] for d in DOCUMENTS)))

---
## Step 3: Create Embeddings

`all-MiniLM-L6-v2` is a free, fast embedding model (384 dimensions). It converts any text into a list of 384 numbers that captures its meaning.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load the embedding model (downloads ~90MB on first run)
print('Loading embedding model...')
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print('✅ Model loaded')

# Test: embed a single sentence
test_text = 'How long does a refund take?'
test_vector = embedder.encode(test_text)

print(f'\nText: "{test_text}"')
print(f'Vector shape: {test_vector.shape}')   # should be (384,)
print(f'First 8 values: {test_vector[:8].round(3)}')

---
## Step 4: Understand Cosine Similarity

Before using a vector DB, let's manually compute similarity to see what's happening.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Embed our sample sentences
sentences = [
    'How long does a refund take?',
    'Refunds are processed in 5 to 7 business days',      # very similar
    'Items must be returned within 30 days',               # related
    'Shipping takes 3 to 5 business days',                # different topic
    'The weather in Tokyo is 22 degrees',                 # completely unrelated
]

vectors = embedder.encode(sentences)
query_vector = vectors[0].reshape(1, -1)  # the question

print('Query: "How long does a refund take?"')
print('\nSimilarity to each sentence:')
print('-' * 55)

for i, sentence in enumerate(sentences[1:], 1):
    sim = cosine_similarity(query_vector, vectors[i].reshape(1, -1))[0][0]
    bar = '█' * int(sim * 30)
    print(f'{sim:.3f} {bar}')
    print(f'       "{sentence[:50]}"')
    print()

---
## Step 5: Store Documents in ChromaDB

ChromaDB is a free, local vector database. It stores your embeddings and lets you search by similarity.

In [ ]:
import chromadb

# Create a local in-memory ChromaDB instance
chroma_client = chromadb.Client()

# Create a collection (like a table in a regular DB)
collection = chroma_client.create_collection(
    name='company_docs',
    metadata={'hnsw:space': 'cosine'}  # use cosine similarity
)

# Embed all documents
print('Embedding and storing documents...')
texts    = [doc['text']   for doc in DOCUMENTS]
ids      = [doc['id']     for doc in DOCUMENTS]
metadatas = [{'source': doc['source']} for doc in DOCUMENTS]

embeddings = embedder.encode(texts).tolist()  # convert numpy → list for ChromaDB

# Add to ChromaDB
collection.add(
    ids=ids,
    embeddings=embeddings,
    documents=texts,
    metadatas=metadatas
)

print(f'✅ Stored {collection.count()} documents in ChromaDB')

---
## Step 6: Query the Vector Database

Now let's search for relevant documents using natural language queries.

In [ ]:
def retrieve(query: str, top_k: int = 2) -> list:
    """
    Search the vector database for the most relevant chunks.
    Returns a list of (text, source, similarity_score) tuples.
    """
    # Embed the query
    query_embedding = embedder.encode(query).tolist()
    
    # Search ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=['documents', 'metadatas', 'distances']
    )
    
    # Format results (distance → similarity: lower distance = more similar)
    retrieved = []
    for text, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        similarity = 1 - dist  # convert distance to similarity score
        retrieved.append({'text': text, 'source': meta['source'], 'score': similarity})
    
    return retrieved

# Test 1
print('Query: "How long does it take to get a refund?"')
print('=' * 55)
for r in retrieve('How long does it take to get a refund?'):
    print(f'Score: {r["score"]:.3f} | Source: {r["source"]}')
    print(f'Text:  {r["text"][:80]}...')
    print()

# Test 2
print('Query: "Is there free shipping?"')
print('=' * 55)
for r in retrieve('Is there free shipping?'):
    print(f'Score: {r["score"]:.3f} | Source: {r["source"]}')
    print(f'Text:  {r["text"][:80]}...')
    print()

---
## Step 7: Full RAG Pipeline (Simulated LLM)

Now we connect retrieval to generation. We simulate the LLM to show the concept clearly.

In [ ]:
def build_rag_prompt(query: str, retrieved_chunks: list) -> str:
    """Build the augmented prompt that grounds the LLM in retrieved context."""
    context_text = '\n\n'.join([
        f'[Source: {r["source"]}]\n{r["text"]}'
        for r in retrieved_chunks
    ])
    
    prompt = f"""You are a helpful customer service assistant.
Answer the user's question ONLY based on the provided context.
If the answer is not in the context, say 'I don't have that information.'

CONTEXT:
{'─' * 50}
{context_text}
{'─' * 50}

USER QUESTION: {query}"""
    return prompt

def simulated_llm_answer(prompt: str) -> str:
    """Simulates an LLM generating an answer (replace with real API call)."""
    # For simulation, we just extract key sentences from context
    lines = [l.strip() for l in prompt.split('.') if len(l.strip()) > 30]
    return f"Based on our documentation: {lines[3] if len(lines) > 3 else lines[-1]}."

def rag_pipeline(query: str, top_k: int = 2) -> str:
    """The complete RAG pipeline: retrieve → augment → generate."""
    print(f'\n{"="*55}')
    print(f'USER: {query}')
    print(f'{"="*55}')
    
    # Step 1: Retrieve relevant chunks
    print('\n[Step 1] Retrieving relevant documents...')
    chunks = retrieve(query, top_k=top_k)
    for i, chunk in enumerate(chunks):
        print(f'  Chunk {i+1} (score={chunk["score"]:.3f}): {chunk["text"][:60]}...')
    
    # Step 2: Build augmented prompt
    print('\n[Step 2] Building augmented prompt...')
    prompt = build_rag_prompt(query, chunks)
    
    # Step 3: Generate answer
    print('\n[Step 3] Generating answer...')
    answer = simulated_llm_answer(prompt)
    print(f'\nAGENT: {answer}')
    return answer

# Test the full pipeline
rag_pipeline('How long does it take to process a refund?')
rag_pipeline('Do you offer free shipping?')
rag_pipeline('What is the CEO salary?')  # should say 'I don\'t have that information'

---
## Step 8: Use a Real LLM for Generation (Groq — Optional)

Get your free key at: **https://console.groq.com**

In [ ]:
!pip install -q groq
from groq import Groq

GROQ_API_KEY = 'your_key_here'  # paste your free key

def rag_pipeline_real_llm(query: str, top_k: int = 2):
    """RAG pipeline with real LLM generation via Groq."""
    client = Groq(api_key=GROQ_API_KEY)
    
    # Step 1: Retrieve
    chunks = retrieve(query, top_k=top_k)
    
    # Step 2: Build context
    context = '\n\n'.join([f'[{r["source"]}] {r["text"]}' for r in chunks])
    
    # Step 3: Generate with grounded prompt
    response = client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[
            {
                'role': 'system',
                'content': (
                    'You are a helpful customer service assistant. '
                    'Answer ONLY based on the provided context. '
                    'If the answer is not in the context, say so.'
                )
            },
            {
                'role': 'user',
                'content': f'Context:\n{context}\n\nQuestion: {query}'
            }
        ],
        temperature=0  # factual retrieval, no creativity needed
    )
    
    answer = response.choices[0].message.content
    print(f'USER:  {query}')
    print(f'AGENT: {answer}')
    print(f'(Grounded from: {[r["source"] for r in chunks]})')
    return answer

if GROQ_API_KEY != 'your_key_here':
    rag_pipeline_real_llm('How long does it take to process a refund?')
else:
    print('Paste your Groq API key above to test with a real LLM.')
    print('Get one free at: https://console.groq.com')

---
## ✅ What You Built

| Component | Purpose |
|---|---|
| `DOCUMENTS` | The raw knowledge base (text chunks) |
| `SentenceTransformer` | Converts text → 384-dim vector |
| `ChromaDB collection` | Stores vectors + text for similarity search |
| `retrieve(query)` | Finds top-K most similar chunks |
| `build_rag_prompt()` | Injects retrieved context into LLM prompt |
| `rag_pipeline()` | Full flow: retrieve → augment → generate |

### The RAG Flow:
```
User Question
      ↓
  Embed query → vector
      ↓
  ChromaDB similarity search → top-K chunks
      ↓
  Inject chunks into prompt
      ↓
  LLM generates grounded answer
      ↓
  User gets factually correct response
```

### Next: Module 5 — Build a Real Agent
We combine Tool Calling (Module 3) + RAG (Module 4) into a complete working agent.